# core

> The source for this library

> This library follows the [fastai style guide](https://docs.fast.ai/dev/style.html).

This is a general purpose library that allows you to use a VLM (Vision Language Model) to thoroughly describe the contents of a video, with subtitles included for context.

The output is a database object containing all the descriptions.

**TODO:**  
- [ ] From populate section onwards, implement the code that builds up to the functions demonstrated in that section.
- [ ] Elaborate on the technical details and put them front and foremost. This page can act as the tecnical blog.
- [ ] Implement parallism to `populate_db`

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

## Database

A database is needed to store information about the:

- videos
- frames each video
- each deployed run
- and the descriptions made for each frame in a run.

So I'll begin by defining the tables.

In [ ]:
#| exports
from fastcore.all import *

In [ ]:
#| exporti
#| echo: true
class Video: id:int; title:str=''; overview:str=''; description:str=''; transcript:str=''; length:int=0; sample_rate:int=0; path:str=''
class Frame: id:int; video_id:int; frame_number:int; subtitle:str=''
class Run: id:int; deploy_time:str; finish_time:str; start_time:str; total_duration:str; video_id:int; model:str; usage:str; num_frames:int; start_sec:int=0; end_sec:int=0; step:int=1; description:str=''
class RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str; ratelimited:bool=False

In [ ]:
#| exports
from fastlite import *

Time to create the database object. For that, I'll be using [fastlite](https://fastlite.answer.ai/) which is a wrapper over [sqlite-utils](https://sqlite-utils.datasette.io/en/stable/).

In [ ]:
!rm db.db
db = database('db.db'); db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

I'll now create the tables from the classes I've defined.

In [ ]:
videos = db.create(Video, transform=True); print(videos.schema)

CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [description] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER,
   [path] TEXT
)


While I can view the schema, I'll add some markdown highlighting to make things easier to visually distinguish. sqlite-utils tables inherit from `Queryable`. So I'll patch its `.schema` method with fastcore's `hl_md` function.

In [ ]:
??Queryable.schema

````python
@property
def schema(self) -> str:
    "SQL schema for this table or view."
    return self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0]
````

**File:** `/usr/local/lib/python3.12/site-packages/apswutils/db.py`; line: 1359

In [ ]:
?hl_md

````python
def hl_md(
    s, lang:str='html', show:bool=True
):
    "Syntax highlight `s` using `lang`."
````

**File:** `~/.local/lib/python3.12/site-packages/fastcore/xtras.py`; line: 1098

**Type:** function

In [ ]:
#| exports
@patch(as_prop=True)
def schema(self:Queryable) -> str:
    "SQL schema for this table or view."
    return hl_md(self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0], lang='sql')

In [ ]:
videos.schema

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [description] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER,
   [path] TEXT
)
```

There, much nicer. I'll create the remaining tables.

In [ ]:
frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')]); frames.schema

```sql
CREATE TABLE [frame] (
   [id] INTEGER PRIMARY KEY,
   [video_id] INTEGER REFERENCES [video]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_number] INTEGER,
   [subtitle] TEXT
)
```

In [ ]:
runs = db.create(Run, transform=True); runs.schema

```sql
CREATE TABLE [run] (
   [id] INTEGER PRIMARY KEY,
   [deploy_time] TEXT,
   [finish_time] TEXT,
   [start_time] TEXT,
   [total_duration] TEXT,
   [video_id] INTEGER,
   [model] TEXT,
   [usage] TEXT,
   [num_frames] INTEGER,
   [start_sec] INTEGER,
   [end_sec] INTEGER,
   [step] INTEGER,
   [description] TEXT
)
```

In [ ]:
runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True); runframes.schema

```sql
CREATE TABLE [run_frame] (
   [run_id] INTEGER REFERENCES [run]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_id] INTEGER REFERENCES [frame]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [type] TEXT,
   [system_prompt] TEXT,
   [prompt] TEXT,
   [description] TEXT,
   [usage] TEXT,
   [ratelimited] INTEGER,
   PRIMARY KEY ([run_id], [frame_id], [type])
)
```

I now wrap everything together that initializes the database in a single function.

In [ ]:
#| exports
from apswutils.db import Database

In [ ]:
#| exports
def init_db(
    path:str|Path='db.db' # Path to database
) -> Database:
    "Initialize a database and return it."
    db = database(path)
    db.create(Video, transform=True)
    db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')])
    db.create(Run, transform=True)
    db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True)
    for t in db.t: t.dataclass()
    return db

In [ ]:
!rm db.db
db = init_db(); db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
db.t.video.schema

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [description] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER,
   [path] TEXT
)
```

I also define a little helper function here that makes it easier to view all rows of a given column in a table.

In [ ]:
#| exports
from operator import attrgetter, itemgetter

In [ ]:
#| exports
def view_col(
    table:Table, # Database table
    col:str, # Column to view
    where:str|None=None, # SQL lookup statement
    where_args:str|None=None # SQL lookup statement arguments
)->list: # List of rows
    "Return all rows of a given column in table, optionally setting `where`."
    if where is None: return L(table()).map(attrgetter(col))
    else: return L(table(where=where, where_args=where_args)).map(itemgetter(col))

## Populate

In this section, I write the logic that will populate the `video` and `frame` tables in one function.

For this to work, the following assumptions are made:
- the provided directory is flat,
- all folders contain the video frames, and
- the same folders contain the respective video's transcript.

In [ ]:
!tree ../../data/timss -L 1

../../data/timss
├── M-AU1
├── M-AU2
├── M-AU3
├── M-AU4
├── M-CZ1
├── M-CZ2
├── M-CZ3
├── M-CZ4
├── M-HK1
├── M-HK2
├── M-HK3
├── M-HK4
├── M-JP1
├── M-JP2
├── M-JP3
├── M-JP4
├── M-NL1
├── M-NL2
├── M-NL3
├── M-NL4
├── M-SW1
├── M-SW2
├── M-SW3
├── M-SW4
├── M-US1
├── M-US2
├── M-US3
└── M-US4

29 directories, 0 files


In [ ]:
dpath = Path('../../data/timss/'); dpath.ls()

[Path('../../data/timss/M-CZ3'), Path('../../data/timss/M-AU2'), Path('../../data/timss/M-CZ4'), Path('../../data/timss/M-JP2'), Path('../../data/timss/M-AU1'), Path('../../data/timss/M-NL2'), Path('../../data/timss/M-AU3'), Path('../../data/timss/M-SW2'), Path('../../data/timss/M-HK4'), Path('../../data/timss/M-HK2'), Path('../../data/timss/.DS_Store'), Path('../../data/timss/M-NL4'), Path('../../data/timss/M-CZ1'), Path('../../data/timss/M-NL1'), Path('../../data/timss/M-US3'), Path('../../data/timss/M-SW3'), Path('../../data/timss/M-US1'), Path('../../data/timss/M-JP4'), Path('../../data/timss/M-US2'), Path('../../data/timss/M-JP3'), Path('../../data/timss/M-CZ2'), Path('../../data/timss/M-HK3'), Path('../../data/timss/M-SW4'), Path('../../data/timss/M-HK1'), Path('../../data/timss/M-JP1'), Path('../../data/timss/M-SW1'), Path('../../data/timss/M-AU4'), Path('../../data/timss/M-NL3'), Path('../../data/timss/M-US4')]

I can see that my data has some unneccessary files, such as `.DS_Store`, I need to filter them out.

In [ ]:
#｜ exports
def filter_paths(
    paths:list[Path], # List of paths to filter
    chs:str='.', # Characters to check for in the component
    comp:str='stem', # Path attribute to inspect (e.g. 'stem', 'name')
    negate:bool=True, # If True, exclude paths whose `comp` contains `chs`; if False, keep only those
)->list[Path]: # Filtered list of paths
    "Filter paths by whether a path component contains specified characters."
    return paths.filter(~getattr(Self, comp).count(chs), negate=negate)

In [ ]:
dpaths = filter_paths(dpath.ls()).sorted(lambda o: (o.stem[:-1], o.stem[-1])); dpaths

[Path('../../data/timss/M-AU1'), Path('../../data/timss/M-AU2'), Path('../../data/timss/M-AU3'), Path('../../data/timss/M-AU4'), Path('../../data/timss/M-CZ1'), Path('../../data/timss/M-CZ2'), Path('../../data/timss/M-CZ3'), Path('../../data/timss/M-CZ4'), Path('../../data/timss/M-HK1'), Path('../../data/timss/M-HK2'), Path('../../data/timss/M-HK3'), Path('../../data/timss/M-HK4'), Path('../../data/timss/M-JP1'), Path('../../data/timss/M-JP2'), Path('../../data/timss/M-JP3'), Path('../../data/timss/M-JP4'), Path('../../data/timss/M-NL1'), Path('../../data/timss/M-NL2'), Path('../../data/timss/M-NL3'), Path('../../data/timss/M-NL4'), Path('../../data/timss/M-SW1'), Path('../../data/timss/M-SW2'), Path('../../data/timss/M-SW3'), Path('../../data/timss/M-SW4'), Path('../../data/timss/M-US1'), Path('../../data/timss/M-US2'), Path('../../data/timss/M-US3'), Path('../../data/timss/M-US4')]

The way I've defined `filter_paths` means I can filter for any file. I'll filter for the transcripts and take a look inside one of them.

In [ ]:
dp = dpaths[0]; dp

Path('../../data/timss/M-AU1')

In [ ]:
tr = filter_paths(dp.ls(), chs='txt', comp='suffix', negate=False)[0]
print(tr.read_text()[:500])

1
00:00:20,000 --> 00:00:42,980
[T] I'm wired.

2
00:00:43,000 --> 00:00:53,980
[T] It's running.

3
00:00:54,000 --> 00:00:56,980
[SN] (inaudible) please turn off the air.

4
00:00:57,000 --> 00:01:06,980
[T] I'll make it a bit warmer.

5
00:01:07,000 --> 00:01:16,980
[T] Well not really, but if you really have to I suppose. Okay.

6
00:01:17,000 --> 00:01:20,980
[SN] (inaudible) go out to my locker and get my maths book?

7
00:01:21,000 --> 00:01:22,980
[T] Oh, you won't need it today.

8
00:0


The transcripts are in SRT format. I'll be converting them to TSV, making them token efficient. To do that, I'll need a way to convert all timestamps to seconds.

In [ ]:
f'45 seconds is {int(45)*60**1} seconds'

'45 seconds is 2700 seconds'

In [ ]:
f'45 minutes is {int(45)*60**2} seconds'

'45 minutes is 162000 seconds'

In [ ]:
f'45 hours is {int(45)*60**3} seconds'

'45 hours is 9720000 seconds'

In [ ]:
'01:50'.split(':')

['01', '50']

In [ ]:
L(enumerate(reversed('01:50'.split(':'))))

[(0, '50'), (1, '01')]

In [ ]:
sum(int(x)*60**i for i,x in enumerate(reversed('01:50'.split(':'))))

110

In [ ]:
#| exports
import re

In [ ]:
#| exports
def time2sec(
    time:str, # Time string in HH:MM:SS or MM:SS format
)->int: # Total seconds
    "Convert a time string to total seconds."
    return sum(int(x)*60**i for i,x in enumerate(reversed(time.split(':'))))

In [ ]:
time2sec('1:50'), time2sec('00:01:30')

(110, 90)

With that sorted, I can now write the function that will convert from SRT to TSV.

In [ ]:
#| exports
def srt2tsv(
    srt:str, # SRT subtitle text to parse
)->str: # TSV string with columns: timerange, speaker, text
    "Convert SRT format to TSV string."
    blocks = re.split(r'\n\n+', srt.strip())
    res = ''
    for b in blocks:
        lines = b.strip().splitlines()
        if len(lines)<3: continue
        parts = lines[1].split(' --> ')
        start = time2sec(parts[0].replace(',', '')[:-3])
        end = time2sec(parts[1].replace(',', '')[:-3])
        m = re.match(r'\[(.*?)\]\s*(.*)', lines[2]); spk,text = (m[1], m[2]) if m else ('', lines[2])
        res += f"{start}→{end}\t{spk}\t{text}\n"
    return res

In [ ]:
print(srt2tsv(tr.read_text())[:500])

20→42	T	I'm wired.
43→53	T	It's running.
54→56	SN	(inaudible) please turn off the air.
57→66	T	I'll make it a bit warmer.
67→76	T	Well not really, but if you really have to I suppose. Okay.
77→80	SN	(inaudible) go out to my locker and get my maths book?
81→82	T	Oh, you won't need it today.
83→97	SN	Oh, okay.
98→98	T	Are you going to be sitting down?
99→100	SN	I haven't (inaudible)
101→102	T	Oh, then you've got to go outside then.
103→105	S	(inaudible) books (inaudible)
106→135	T	There'll be some


I can now try create an entry in the `video` table.

In [ ]:
dt = db.t; dt

frame, run, run_frame, video

In [ ]:
dt.video.dataclass(); dt.frame.dataclass()

fastlite.core.Frame

In [ ]:
v = dt.video.insert(title=dp.stem, transcript=srt2tsv(tr.read_text()),
                   length=len(dp.ls()), path=str(dp), sample_rate=1)
type(v), v.transcript[:50]

(fastlite.core.Video,
 "20→42\tT\tI'm wired.\n43→53\tT\tIt's running.\n54→56\tSN\t")

Now I want to attempt creating some entries in the `frame` table. For that, I'll have to fetch the corresponding subtitle for each frame.

In [ ]:
db.t.frame.schema

```sql
CREATE TABLE [frame] (
   [id] INTEGER PRIMARY KEY,
   [video_id] INTEGER REFERENCES [video]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_number] INTEGER,
   [subtitle] TEXT
)
```

In [ ]:
dp

Path('../../data/timss/M-AU1')

In [ ]:
dp.ls()[:5]

[Path('../../data/timss/M-AU1/frame_001886.jpg'), Path('../../data/timss/M-AU1/frame_001303.jpg'), Path('../../data/timss/M-AU1/frame_001378.jpg'), Path('../../data/timss/M-AU1/frame_002173.jpg'), Path('../../data/timss/M-AU1/frame_002259.jpg')]

In [ ]:
filter_paths(dp.ls(), chs='.', comp='stem')[:3]

[Path('../../data/timss/M-AU1/frame_001886.jpg'), Path('../../data/timss/M-AU1/frame_001303.jpg'), Path('../../data/timss/M-AU1/frame_001378.jpg')]

In [ ]:
filter_paths(dp.ls(), chs='txt', comp='suffix')[:3]

[Path('../../data/timss/M-AU1/frame_001886.jpg'), Path('../../data/timss/M-AU1/frame_001303.jpg'), Path('../../data/timss/M-AU1/frame_001378.jpg')]

To connect each frame to their corresponding transcript, I'll be converting the TSV transcripts to Python dictionaries. If the video had a subtitle between 3 seconds and 7 seconds. Then all frames sampled in that range should have the corresponding same subtitle.

In [ ]:
#| exports
def tsv2dict(
    tsv:str, # TSV string with columns: timerange, speaker, text
    max_len:int, # Max frame number to cover
    default:str='', # Value for timestamps without subtitles
)->dict: # {second: "speaker: text"} lookup dict
    "Convert TSV string to a {second: subtitle} lookup dict."
    lookup = {}
    for line in tsv.splitlines():
        if not line.strip(): continue
        tr, spk, text = line.split('\t')
        start, end = tr.split('→')
        start, end = int(start), int(end)
        entry = f"{spk}: {text}" if spk else text
        for s in range(start, end+1): lookup[s] = entry
    end = max(max(lookup) if lookup else 0, max_len)
    for s in range(end+1): lookup.setdefault(s, default)
    return lookup

In [ ]:
r = srt2tsv(tr.read_text())

Doule checking whether I got what I wanted.

In [ ]:
from itertools import islice

In [ ]:
list(islice(tsv2dict(r, 2000).items(), 10))

[(20, "T: I'm wired."),
 (21, "T: I'm wired."),
 (22, "T: I'm wired."),
 (23, "T: I'm wired."),
 (24, "T: I'm wired."),
 (25, "T: I'm wired."),
 (26, "T: I'm wired."),
 (27, "T: I'm wired."),
 (28, "T: I'm wired."),
 (29, "T: I'm wired.")]

In [ ]:
lookup = tsv2dict(srt2tsv(tr.read_text()), len(dp.ls()))
lookup[20], lookup[100]

("T: I'm wired.", "SN: I haven't (inaudible)")

I can go ahead an produce an entry.

In [ ]:
fpaths = filter_paths(Path(v.path).ls(), chs='txt', comp='suffix')
fpaths = filter_paths(fpaths, chs='.', comp='stem').sorted(key=~Self.stem.split('_'))
fpaths[:5]

[Path('../../data/timss/M-AU1/frame_000001.jpg'), Path('../../data/timss/M-AU1/frame_000002.jpg'), Path('../../data/timss/M-AU1/frame_000003.jpg'), Path('../../data/timss/M-AU1/frame_000004.jpg'), Path('../../data/timss/M-AU1/frame_000005.jpg')]

In [ ]:
fp = fpaths[0]; fp

Path('../../data/timss/M-AU1/frame_000001.jpg')

In [ ]:
fnum = int(fp.stem.split('_')[1]); fnum

1

In [ ]:
db.t.frame.insert(video_id=v.id, frame_number=fnum, subtitle=tsv2dict(v.transcript, v.length)[fnum])

Frame(id=1, video_id=1, frame_number=1, subtitle='')

I can wrap this all up into a function that will perform this on all videos and all frames.

In [ ]:
# #| exports
# from fastprogress.fastprogress import NBMasterBar as master_bar, NBProgressBar as progress_bar

In [ ]:
#| exports
from fastprogress.fastprogress import master_bar, progress_bar

In [ ]:
#| exports
def populate_db(
    db:Database, # Database to populate
    paths:list[Path], # List of video directories
    sample_rate:int=1, # Sampling rate for frames
    trans_suffix:str='txt', # Transcript file suffix
)->None:
    "Populate video and frame tables from a list of video directories."
    t = db.t
    filter_trans = partial(filter_paths, chs=trans_suffix, comp='suffix', negate=False)
    filter_dots = partial(filter_paths, chs='.', comp='stem')
    for p in (mb:=master_bar(paths)):
        mb.main_bar.comment = f'video {p.stem}'
        tr = filter_trans(p.ls())[0]
        v = t.video.insert(title=p.stem, transcript=srt2tsv(tr.read_text()),
                           length=len(p.ls()), path=str(p), sample_rate=sample_rate)
        lookup = tsv2dict(v.transcript, v.length)
        fpaths = filter_dots(filter_paths(Path(v.path).ls(), chs=trans_suffix, comp='suffix')).sorted(key=~Self.stem.split('_'))
        for fp in (pb:=progress_bar(fpaths, parent=mb)):
            fnum = int(fp.stem.split('_')[1])
            t.frame.insert(video_id=v.id, frame_number=fnum, subtitle=lookup[fnum])

In [ ]:
!rm db.db
db = init_db()
populate_db(db, dpaths)

Epoch 1/28 : |----------------------------------------| 0.00% [0/2639 00:00<?]

Epoch 1/28 : |----------------------------------------| 0.04% [1/2639 00:00<00:03]

Epoch 1/28 : |----------------------------------------| 0.08% [2/2639 00:00<00:02]

Epoch 1/28 : |----------------------------------------| 0.11% [3/2639 00:00<00:02]

Epoch 1/28 : |----------------------------------------| 0.15% [4/2639 00:00<00:02]

...

Epoch 28/28 : |█████████████---------------------------| 33.58% [905/2695 00:00<00:00]

Epoch 28/28 : |█████████████████████-------------------| 52.88% [1425/2695 00:00<00:00]

Epoch 28/28 : |████████████████████████████------------| 72.02% [1941/2695 00:00<00:00]

Epoch 28/28 : |████████████████████████████████████----| 91.17% [2457/2695 00:00<00:00]

Epoch 28/28 : |████████████████████████████████████████| 100.00% [2695/2695 00:01<00:00]

In [ ]:
len(db.t.video()), len(db.t.frame())

(28, 80308)

## VLM

In this section I create helper functions for working with the LLM, using [fastllm](https://github.com/AnswerDotAI/fastllm) library.

In [ ]:
from aidialog.msg_parts import Msg, Part, PartType

In [ ]:
?Msg

````python
def Msg(
    role:str, content:List
)->None:
    "A normalized message."
````

**File:** `/usr/local/lib/python3.12/site-packages/aidialog/msg_parts.py`; line: 55

**Type:** type

In [ ]:
?Part

````python
def Part(
    type:str, text:str=None, data:dict=None
)->None:
    "A normalized content part."
````

**File:** `/usr/local/lib/python3.12/site-packages/aidialog/msg_parts.py`; line: 21

**Type:** type

In [ ]:
?Part

````python
def Part(
    type:str, text:str=None, data:dict=None
)->None:
    "A normalized content part."
````

**File:** `/usr/local/lib/python3.12/site-packages/aidialog/msg_parts.py`; line: 21

**Type:** type

In [ ]:
def user(
    txt:str,
    img:str|None=None,
)->Msg:
    "Build a user message with optional image."
    if img is None: return Msg(role='user', content=[Part(PartType.text, text=txt)])
    else: return Msg(role='user', content=[Part(PartType.input_image, text=img), Part(PartType.text, text=txt)])

In [ ]:
user('你好')

**Msg**

- role: `user`

<contents>

**Part** (`text`)

你好

<details markdown='1'>

- data: `None`

</details>

</contents>

In [ ]:
def assistant(
    txt:str,
    citations:list=None,
)->Msg:
    "Build an assistant message."
    return Msg(role='assistant', content=[Part(PartType.text, text=txt, data={'citations': citations} if citations else None)])

In [ ]:
assistant('嗨')

**Msg**

- role: `assistant`

<contents>

**Part** (`text`)

嗨

<details markdown='1'>

- data: `None`

</details>

</contents>

In [ ]:
from fastllm.acomplete import acomplete

In [ ]:
?acomplete

````python
async def acomplete(
    msgs, model, api_name:NoneType=None, vendor_name:NoneType=None, api_key:NoneType=None, base_url:NoneType=None,
    xtra_body:NoneType=None, xtra_hdrs:NoneType=None, stream:bool=False, stop_callables:NoneType=None, retries:int=2,
    retry_delay:float=0.5, system:NoneType=None, max_tokens:NoneType=None, temperature:NoneType=None,
    tools:NoneType=None, tool_choice:NoneType=None, reasoning_effort:NoneType=None, web_search_options:NoneType=None
):
    "Unified completion across different APIs."
````

**File:** `/usr/local/lib/python3.12/site-packages/fastllm/acomplete.py`; line: 170

**Type:** function

In [ ]:
from cachy import enable_cachy, disable_cachy

[Cachy](https://github.com/AnswerDotAI/cachy) caches http requests and stores the responses locally. This reduces spend and avoids the need to also repeatedly wait for the remote server to process my request.


In [ ]:
enable_cachy()
await acomplete([user('hi')], 'deepseek-v4-flash', vendor_name='deepseek')

<details><summary>Thinking</summary>

好的，用户只发了一个“hi”，这是非常简单的打招呼。用户可能刚进入对话，想测试我是否在线或者开始一个友好的交流。深层需求应该是希望得到热情、友好的回应，开启一次对话。我不需要复杂分析，直接礼貌问候并表达乐于助人的态度，用开放式的邀请让用户提出具体问题。想到了用“你好！”开头，加上表情符号显得亲切，然后自我介绍并说明能力范围，最后用提问引导对话继续。

</details>

你好！很高兴见到你！😊

有什么我可以帮你的吗？无论是回答问题、帮你整理信息、提供创作灵感，还是聊聊天，我都很乐意陪你一起。你只需告诉我需求，剩下的交给我！

<details markdown='1'>

- model: `deepseek-v4-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=5, completion_tokens=143, total_tokens=148, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=97, raw={'prompt_tokens': 5, 'completion_tokens': 143, 'total_tokens': 148, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5})`

</details>

In [ ]:
from fastllm.types import Completion

I'm now creating a function that'll stream in the response. We'll see the response here as it generates.

In [ ]:
async def stream(
    msgs:list|None=None,  # Messages to send
    model:str='',  # Model name (e.g. 'deepseek-v4-flash')
    max_think:float=float('inf'),  # Max thinking tokens to display
    usage:bool=True,  # Show usage info in output
    display:bool=True,  # Print text/thinking as it arrives
    **kwargs,  # Passed to acomplete
) -> Completion:  # Return the final completion
    "Stream a response, printing text/thinking as it arrives. Returns the final completion."
    assert msgs is not None, 'no messages provided'
    assert model!='', 'no model name provided'
    think_cnt, seen_txt = 0, False
    async for o in await acomplete(msgs, model, stream=True, **kwargs):
        if not isinstance(o, Completion) and display:
            if isinstance(o, Part) and o.type==PartType.thinking and think_cnt<max_think: print('🤔', end='', flush=True)
            if isinstance(o, Part) and o.type==PartType.text and (txt:=o.text): print(f"{'\n\n' if not seen_txt else ''}{txt}", end='', flush=True) or not seen_txt and (seen_txt:=True)
            think_cnt+=1
    if display: print()
    return o

In [ ]:
r = await stream([user('hi')], 'deepseek-v4-flash', vendor_name='deepseek')

In [ ]:
from base64 import b64encode

And a helper function to more easily pass images to the VLM.

In [ ]:
def img2b64(
    path:Path,
)->str:
    "Encode an image file as a base64 data URL."
    return 'data:image/png;base64,'+b64encode(Path.read_bytes(path)).decode()

In [ ]:
r = await stream([user('what do ye elf eyes see', img2b64(Path('./test.jpg')))], 'bytedance-seed/seed-2.0-lite', vendor_name='openrouter', reasoning_effort='high')

And a helper function to create sessions more easily without having to write the same params each time.

In [ ]:
@delegates(stream, keep=True)
def session(
    **kwargs,
):
    "Create a stream partial with preset model/kwargs."
    return partial(stream, **kwargs)

## Run

In [ ]:
from fastspec.errors import APIError

In [ ]:
async def _process_frame(p:Path, db, session, run_id:int, video_id:int, prompt:str, prompt_type:str, include_subs:bool=True):
    "Run VLM on a single frame and store the result."
    fnum = int(p.stem.split('_')[1])
    frame = db.t.frame.selectone('video_id=? AND frame_number=?', (video_id, fnum))
    if include_subs:
        sub = frame.subtitle or '[No speech]'
        prompt = f"{prompt}\n\nSubtitle: {sub}"
    try:
        r = await session([user(prompt, img=img2b64(p))])
        db.t.run_frame.upsert(run_id=run_id, frame_id=frame.id, type=prompt_type, prompt=prompt, description=r.message.text, usage=r.usage.raw, ratelimited=False)
    except APIError as e:
        sc = getattr(e,'status_code',None)
        if sc==429:
            db.t.run_frame.upsert(run_id=run_id, frame_id=frame.id, type=prompt_type, prompt=prompt, description='', ratelimited=True)
            print(f'!! Rate limited: frame {fnum}')
        elif sc==402:
            print(f'!! Insufficient credits — stopped at frame {fnum}')
            raise
        else: raise

In [ ]:
async def _run_batch(frames:L, db, session, run_id:int, video_id:int, prompt:str, prompt_type:str, include_subs:bool, n_workers:int, pause:float):
    "Run _process_frame across a batch of frame paths in parallel."
    done = 0
    async for i,r in parallel_async_gen(_process_frame, frames, db, session, run_id, video_id,
                                         prompt, prompt_type, include_subs,
                                         n_workers=n_workers, pause=pause):
        done += 1; print(f'\r{done}/{len(frames)}', end='', flush=True)
    print()

In [ ]:
async def _retry_ratelimited(db, video_path:Path, run_id:int, session, prompt:str, prompt_type:str, include_subs:bool, n_workers:int, pause:float, max_retries:int=2):
    "Retry rate-limited frames up to max_retries times."
    for attempt in range(max_retries):
        rl_rows = L(db.t.run_frame('run_id=? AND ratelimited=1', (run_id,)))
        if not rl_rows: break
        rl_fids = rl_rows.map(itemgetter('frame_id'))
        ph = ','.join('?'*len(rl_fids))
        rl_frames = L(db.t.frame(f'id IN ({ph})', tuple(rl_fids)))
        rl_paths = rl_frames.map(lambda f: video_path/f'frame_{f["frame_number"]:06d}.jpg')
        print(f'!! Retrying {len(rl_paths)} rate-limited frames (attempt {attempt+1}/{max_retries})')
        try: await _run_batch(rl_paths, db, session, run_id, video_id, prompt, prompt_type, include_subs, n_workers, pause)
        except APIError as e:
            if getattr(e,'status_code',None)!=402: raise
            break

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
tz = ZoneInfo('Asia/Hong_Kong')

In [ ]:
def _run_header(run, model:str, start:int, stop:int, step:int, cache:bool):
    "Print run header box."
    print(f'╭─ Run #{run.id} ═══════════════════════════════╮\n│ Model    {model}\n│ Start    {start}\n│ Stop     {stop}\n│ Step     {step}\n│ Frames   {(stop-start)//step}\n│ Cache    {cache}\n│ Time     {datetime.now(tz).strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
def compute_usage(db, run_id:int)->str:
    "Aggregate usage stats across all frames in a run. Returns JSON string."
    usgs = L(db.t.run_frame('run_id=? AND ratelimited=0', (run_id,))).map(lambda r: loads(r.usage))
    if not usgs: return '{}'
    tot = {k: ({k2:0 for k2 in v} if isinstance(v,dict) else 0) for k,v in usgs[0].items()}
    for u in usgs:
        for k,v in u.items():
            if isinstance(v,dict):
                for k2,v2 in v.items(): tot[k][k2] += v2
            else: tot[k] += v
    return dumps(tot)

def _finish_run(db, run_id:int)->Run:
    "Update run with finish time and usage, print summary box."
    finish = datetime.now(tz)
    run = db.t.run[run_id]
    start = datetime.fromisoformat(run.deploy_time)
    elapsed = finish - start
    db.t.run.update(id=run_id, finish_time=finish, start_time=start.isoformat(), total_duration=str(elapsed).split('.')[0], usage=compute_usage(db, run_id))
    run = db.t.run[run_id]
    tot = dict2obj(loads(run.usage))
    print(f'╭─ Run #{run.id} Complete ═════════════════════╮\n│ Finish   {datetime.fromisoformat(run.finish_time).strftime("%H:%M:%S")}\n│ Elapsed  {str(elapsed).split(".")[0]}\n│ Cost     ${tot.cost:.4f} (HKD {tot.cost*7.84:.2f})\n╰──────────────────────────────────────────────╯')
    return run

In [ ]:
#| export
def _run_header(run, model:str, start:int, stop:int, step:int, cache:bool):
    "Print run header box."
    print(f'╭─ Run #{run.id} ═══════════════════════════════╮\n│ Model    {model}\n│ Start    {start}\n│ Stop     {stop}\n│ Step     {step}\n│ Frames   {(stop-start)//step}\n│ Cache    {cache}\n│ Time     {datetime.now(tz).strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

In [ ]:
from typing import Callable
from fastprogress.fastprogress import NBMasterBar as master_bar

In [ ]:
from fastcore.parallel import parallel_async_gen
async def deploy_run(
    video_id:int,
    db:Database,
    session:Callable,
    prompt:str,
    prompt_type:str,
    start:int=0,
    stop:int|None=None,
    step:int=1,
    cache:bool=False,
    include_subs:bool=True,
    n_workers:int=8,
    pause:float=3,
    max_retries:int=2,
)->Run:
    "Run a single prompt across a range of frames, storing results in the database."
    video = db.t.video[video_id]
    fpath = L(Path(video.path).glob('frame_*.jpg')).sorted(key=~Self.stem.split('_'))
    if stop is None: stop = len(fpath)
    if not cache: disable_cachy(); print('!! Cache disabled')
    else: print('!! Using cache')

    run = db.t.run.insert(deploy_time=datetime.now(tz), video_id=video_id, model=session.keywords['model'], num_frames=(stop-start)//step, start_sec=start, end_sec=stop, step=step)
    _run_header(run, session.keywords['model'], start, stop, step, cache)

    frames = fpath[start:stop:step]
    try: await _run_batch(frames, db, session, run.id, video_id, prompt, prompt_type, include_subs, n_workers, pause)
    except APIError as e:
        if getattr(e,'status_code',None)!=402: raise

    await _retry_ratelimited(db, Path(video.path), run.id, session, prompt, prompt_type, include_subs, n_workers, pause, max_retries)

    if not cache: enable_cachy(); print('!! Cache enabled')
    return _finish_run(db, run.id)


## Summary

In [ ]:
#| export
from tiktoken import encoding_for_model

In [ ]:
#| export
def _get_runframes(
    db:Database,
    run_ids:int|list[int],
    start:int|None=None,
    stop:int|None=None,
)->L:
    "Query runframes for given run(s), join with frame table for frame_number, group by frame_number."
    if isinstance(run_ids, int): run_ids = [run_ids]
    ph = ','.join('?' * len(run_ids))
    rows = L(db.t.run_frame(f'run_id IN ({ph}) AND ratelimited=0', tuple(run_ids)))
    fids = sorted(set(r.frame_id for r in rows))
    fn_map = {f.id: f.frame_number for f in db.t.frame(f'id IN ({",".join("?"*len(fids))})', tuple(fids))}
    grouped = rows.groupby(lambda r: fn_map[r.frame_id])
    fnums = sorted(grouped.keys())
    if start is not None: fnums = [f for f in fnums if f >= start]
    if stop is not None: fnums = [f for f in fnums if f <= stop]
    return L((fn, L(grouped[fn])) for fn in fnums)

In [ ]:
?encoding_for_model

````python
def encoding_for_model(
    model_name:str
)->Encoding:

````

````
Returns the encoding used by a model.

Raises a KeyError if the model name is not recognised.
````

**File:** `/usr/local/lib/python3.12/site-packages/tiktoken/model.py`; line: 113

**Type:** function

In [ ]:
#| export
def _build_window(
    runframes,
    step:int=1,
)->str:
    "Build window text from grouped runframes."
    window = ''
    for i,rf in runframes[::step]:
        prefix = f'TIMESTAMP {i}s\n'
        window += prefix+len(prefix.strip())*'='+'\n'
        for r in rf:
            window += '\n--\n'+f'{r.type.upper()}\n{r.description}'+'\n\n'
    return window

In [ ]:
#| export
def _summary_header(run_id:int, start:int, stop:int, step:int, model:str, window:str, win_tokens:int, cache:bool, t0:datetime):
    "Print summary run header box."
    print(f'╭─ Summary Run #{run_id} ═══════════════════════╮\n│ Frames   {start}–{stop} (step {step})\n│ Model    {model}\n│ Window   {len(window)} chars / {win_tokens} tokens\n│ Cache    {cache}\n│ Start    {t0.strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
def _summary_footer(summary:str, win_tokens:int, t0:datetime, t1:datetime, cost:float):
    "Print summary completion box."
    enc = encoding_for_model('gpt-4o')
    sum_tokens = len(enc.encode(summary))
    reduction = (1 - sum_tokens/win_tokens)*100 if win_tokens else 0
    elapsed = t1 - t0
    print(f'╭─ Summary Complete ═══════════════════════════╮\n│ Finish   {t1.strftime("%H:%M:%S")}\n│ Elapsed  {str(elapsed).split(".")[0]}\n│ Summary  {len(summary)} chars / {sum_tokens} tokens\n│ Reduced  {reduction:.1f}%\n│ Cost     ${cost:.4f} (HKD {cost*7.84:.2f})\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
async def summarize_window(
    db:Database,
    run_ids:int|list[int],
    start:int,
    stop:int,
    session:Callable,
    sys_prompt:str,
    step:int=1,
    cache:bool=False,
    context:str='',
)->str:
    "Summarize a window of frames from runframes. Returns summary text."
    if not cache: disable_cachy()
    rfs = _get_runframes(db, run_ids, start, stop)
    window = _build_window(rfs, step)
    if context: window = f'PRIOR CONTEXT\n{"="*13}\n{context}\n\n{window}'
    win_tokens = len(encoding_for_model('gpt-4o').encode(window))
    t0 = datetime.now(tz)
    label = run_ids if isinstance(run_ids, int) else f'{run_ids[0]}+{len(run_ids)-1}more'
    _summary_header(label, start, stop, step, session.keywords['model'], window, win_tokens, cache, t0)
    r = await session([user(window)], system=sys_prompt)
    t1 = datetime.now(tz)
    if not cache: enable_cachy()
    summary = r.message.text
    _summary_footer(summary, win_tokens, t0, t1, r.usage.raw.get('cost', 0))
    return summary

In [ ]:
#| export
from IPython.display import clear_output

async def summarize_run(
    db:Database,
    session:Callable,
    sys_prompt:str,
    run_id:int|None=None,
    video_id:int|None=None,
    window_sec:int=300,
    step:int=1,
    cache:bool=False,
)->str:
    "Summarize a single run or all frames for a video in rolling windows."
    if run_id is not None: rids = [run_id]
    elif video_id is not None: rids = L(db.t.run('video_id=?', (video_id,))).map(lambda r: r.id)
    else: raise ValueError('Either run_id or video_id required')
    rfs = _get_runframes(db, rids)
    if not rfs: return ''
    fnums = rfs.itemgot(0)
    full_summary = ''
    chunks = list(chunked(fnums, window_sec))
    for chunk in (mb:=master_bar(chunks)):
        mb.main_bar.comment = f'window {chunk[0]}–{chunk[-1]}s'
        summary = await summarize_window(db, rids, chunk[0], chunk[-1], session, sys_prompt, step=step, cache=cache, context=full_summary)
        heading = f'[{chunk[0]}–{chunk[-1]}s]'
        summary = summary.strip()
        full_summary = full_summary + f'\n\n{heading} {summary}' if full_summary else f'{heading} {summary}' if summary else ''
        clear_output(wait=True)
        print(full_summary)
    if run_id is not None: db.t.run.update(id=run_id, description=full_summary)
    else: db.t.video.update(id=video_id, description=full_summary)
    return full_summary

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()